In [28]:
import numpy as np
import pandas as pd

# Edit the path to your dataset
path = "../../../../assets/datasets/1_UCI_Chronic_Kidney_Disease_Dataset/data/chronic_kidney_disease.csv"

# Edit the name of the target column
target = "class"

# Edit the null value symbol 
df = pd.read_csv(path, na_values=["?"])
df.head()

ParserError: Error tokenizing data. C error: Expected 25 fields in line 71, saw 26


In [ ]:
def column_type(s):
    n = s.nunique()
    if pd.api.types.is_bool_dtype(s):
        return "boolean"
    if pd.api.types.is_numeric_dtype(s):
        if n == 2:
            return "numeric (binary)"
        if pd.api.types.is_integer_dtype(s):
            return "numeric (integer)"
        return "numeric (float)"
    if n <= 20 or n / max(len(s), 1) < 0.05:
        return "categorical (string)"
    return "string (free text / id)"


def has_classes(s, max_discrete=10):
    kind = column_type(s)
    if kind in ("boolean", "numeric (binary)") or kind.startswith("categorical"):
        return True
    return kind == "numeric (integer)" and s.nunique() <= max_discrete

In [ ]:
rows, cols = df.shape
y = df[target]
valid_rows = len(df.dropna())

summary = pd.DataFrame({
    "type": pd.Series({c: column_type(df[c]) for c in df.columns}),
    "unique": df.nunique(),
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
})

summary["classes"] = pd.Series({
    c: sorted(df[c].dropna().unique().tolist(), key=str) if has_classes(df[c]) else "-"
    for c in df.columns
})

In [ ]:
print("DATASET OVERVIEW")
print(f"Instances (rows):            {rows}")
print(f"Columns:                     {cols}")
print(f"Features (excl. target):     {cols - 1}")
print(f"Valid instances (no NaN):    {valid_rows} ({valid_rows / rows:.2%})")
print(f"Instances with any NaN:      {rows - valid_rows}")
print(f"Duplicate rows:              {df.duplicated().sum()}")
print(f"Memory:                      {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
print(f"Features per instance ratio: {(cols - 1) / rows:.4f}")

In [ ]:
print("COLUMN TYPES AND CLASSES")
print(summary.to_string())

print("\nTYPE COUNTS")
print(summary["type"].value_counts().to_string())

constant = summary.index[summary["unique"] <= 1].tolist()
print(f"\nConstant columns: {constant if constant else 'none'}")

In [ ]:
print(f"TARGET: {target}")
if has_classes(y) or not pd.api.types.is_numeric_dtype(y):
    counts = y.value_counts(dropna=False)
    share = (counts / len(y) * 100).round(2)
    print("Task:                  classification")
    print(f"Number of classes:     {y.nunique()}")
    print(f"Class values:          {sorted(y.dropna().unique().tolist(), key=str)}")
    print(pd.DataFrame({"count": counts, "%": share}).to_string())
    print(f"Imbalance ratio (max/min): {counts.max() / counts.min():.2f}")
    print(f"Majority-class baseline:   {counts.max() / len(y):.2%}")
else:
    print("Task:                  regression")
    print(y.describe().to_string())

In [ ]:
numeric = df.select_dtypes(include=[np.number])
if not numeric.empty:
    print("NUMERIC FEATURE STATS")
    stats = numeric.describe().T[["min", "max", "mean", "std"]]
    stats["skew"] = numeric.skew()
    print(stats.round(3).to_string())